# MAIN-1: Ensembl vs CAT Annotation Concordance Sankey

Hierarchical Sankey/alluvial diagram showing progressive concordance between
Ensembl (anchor-sequence projection) and CAT (graph-based projection) annotations.

**Levels:**
1. Gene presence: Present in both | Ensembl only | CAT only
2. RBH status: RBH found | No RBH (subset of 'both')
3. Transcript concordance: Full | Partial | None (subset of 'RBH found')
4. CDS integrity: Intact | Partial | Disrupted (subset of coding RBH)

**Input:** `intermediate_spreadsheets/sankey/` from workflow

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.sankey import Sankey
import numpy as np
from pathlib import Path

# Configuration
RESULTS_DIR = Path('../results')  # Adjust to your pipeline output directory
SANKEY_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'sankey'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colour scheme
COLORS = {
    'high_confidence': '#1b4f72',   # Dark blue
    'concordant': '#2e86c1',        # Medium blue
    'partial': '#85c1e9',           # Light blue
    'discordant': '#e74c3c',        # Red
    'ensembl_only': '#f39c12',      # Orange
    'cat_only': '#e67e22',          # Dark orange
    'background': '#ecf0f1',        # Light grey
}

In [ ]:
# Load pre-computed Sankey data
per_asm = pd.read_csv(SANKEY_DIR / 'sankey_per_assembly_flows.tsv', sep='\t')
flow_counts = pd.read_csv(SANKEY_DIR / 'sankey_flow_counts.tsv', sep='\t')

print(f"Loaded data for {len(per_asm)} assemblies")
print(f"\nAggregate flow counts:")
display(flow_counts.T)

In [ ]:
# Compute median flows for the Sankey diagram
medians = {
    # Named genes only (non-ENSG): gene name concordance
    'l1_named_both': per_asm['l1_named_both'].median(),
    'l1_named_ens_only': per_asm['l1_named_ensembl_only'].median(),
    'l1_named_cat_only': per_asm['l1_named_cat_only'].median(),
    # RBH: all gene pairs, own denominator (independent of named gene presence)
    'l2_pass': per_asm['l2_rbh_pass'].median(),
    'l2_fail': per_asm['l2_rbh_fail'].median(),
    'l2_total': per_asm['l2_rbh_total'].median(),
    # Transcript concordance (subset of RBH pass)
    'l3_full': per_asm['l3_tx_full'].median(),
    'l3_partial': per_asm['l3_tx_partial'].median(),
    'l3_none': per_asm['l3_tx_none'].median(),
    # Biotype breakdown of fully concordant genes
    'l3b_protein_coding': per_asm['l3b_protein_coding'].median(),
    'l3b_lncrna': per_asm['l3b_lncrna'].median(),
    'l3b_pseudogene': per_asm['l3b_pseudogene'].median(),
    'l3b_other_ncrna': per_asm['l3b_other_ncrna'].median(),
    'l3b_other': per_asm['l3b_other'].median(),
    # CDS integrity (subset of coding full-concordant genes)
    'l4_intact': per_asm['l4_cds_intact'].median(),
    'l4_partial': per_asm['l4_cds_partial'].median(),
    'l4_disrupted': per_asm['l4_cds_disrupted'].median(),
}

# Print headline concordance numbers
named_total = medians['l1_named_both'] + medians['l1_named_ens_only'] + medians['l1_named_cat_only']
print(f"Median named gene concordance: {medians['l1_named_both']/named_total*100:.1f}%  "
      f"(both={int(medians['l1_named_both']):,}, "
      f"ens_only={int(medians['l1_named_ens_only']):,}, "
      f"cat_only={int(medians['l1_named_cat_only']):,})")
print(f"Median RBH pass rate: {per_asm['l2_pct_pass'].median():.1f}%  "
      f"(pass={int(medians['l2_pass']):,} / total={int(medians['l2_total']):,})")
print(f"Median transcript concordance (of RBH pass): {per_asm['l3_pct_full'].median():.1f}%")
l3b_total = sum(medians[k] for k in ['l3b_protein_coding', 'l3b_lncrna', 'l3b_pseudogene', 'l3b_other_ncrna', 'l3b_other'])
print(f"  → Fully concordant biotype breakdown: "
      f"protein_coding={int(medians['l3b_protein_coding']):,}  "
      f"lncRNA={int(medians['l3b_lncrna']):,}  "
      f"pseudogene={int(medians['l3b_pseudogene']):,}  "
      f"other_ncRNA={int(medians['l3b_other_ncrna']):,}  "
      f"other={int(medians['l3b_other']):,}")
print(f"Median CDS integrity (of coding full-concordant): {per_asm['l4_pct_intact'].median():.1f}%")

In [ ]:
# --- MAIN-1 Figure: 5-level stacked bar chart + flow ribbons ---
# Principle: each level's denominator is the "winning" subset of the level above.
#
#  L1  Named gene presence        denominator: all named genes
#  L2  RBH locus overlap          denominator: all RBH pairs (own)
#  L3  Transcript concordance     denominator: L2 RBH pass
#  L3b Biotype of fully concordant denominator: L3 full match
#  L4  CDS integrity              denominator: L3b protein-coding

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

def _segment_ranges(labels, pcts):
    out = {}
    left = 0.0
    for lab, pct in zip(labels, pcts):
        out[lab] = (left, left + float(pct))
        left += float(pct)
    return out

def _draw_ribbons_bottom_to_top(
    ax, y_upper_center, y_lower_center, bar_height,
    upper_labels, upper_pcts, lower_labels, lower_pcts,
    flow_counts, upper_color_map,
    alpha=0.28, zorder=0.7, y_gap=0.02,
):
    upper_ranges = _segment_ranges(upper_labels, upper_pcts)
    lower_ranges = _segment_ranges(lower_labels, lower_pcts)
    upper_cursor = {u: upper_ranges[u][0] for u in upper_labels}
    lower_cursor = {l: lower_ranges[l][0] for l in lower_labels}

    upper_den = {u: sum(flow_counts.get(u, {}).values()) for u in upper_labels}
    lower_den = {l: sum(flow_counts.get(u, {}).get(l, 0) for u in upper_labels) for l in lower_labels}

    def w_upper(u, c):
        seg_w = upper_ranges[u][1] - upper_ranges[u][0]
        return (c / upper_den[u]) * seg_w if upper_den[u] > 0 else 0.0

    def w_lower(l, c):
        seg_w = lower_ranges[l][1] - lower_ranges[l][0]
        return (c / lower_den[l]) * seg_w if lower_den[l] > 0 else 0.0

    y_upper_bottom = y_upper_center - bar_height / 2 + y_gap
    y_lower_top    = y_lower_center + bar_height / 2 - y_gap

    for u in upper_labels:
        for l in lower_labels:
            c = float(flow_counts.get(u, {}).get(l, 0) or 0)
            if c <= 0:
                continue
            x0 = upper_cursor[u];  x1 = x0 + w_upper(u, c);  upper_cursor[u] = x1
            x2 = lower_cursor[l];  x3 = x2 + w_lower(l, c);  lower_cursor[l] = x3
            ax.add_patch(Polygon(
                [(x0, y_upper_bottom), (x1, y_upper_bottom), (x3, y_lower_top), (x2, y_lower_top)],
                closed=True, facecolor=upper_color_map.get(u, '#999999'),
                edgecolor='none', alpha=alpha, zorder=zorder, clip_on=True,
            ))

# ── Build level data ──────────────────────────────────────────────────────────

named_total = medians['l1_named_both'] + medians['l1_named_ens_only'] + medians['l1_named_cat_only']
l1_vals   = [medians['l1_named_both'], medians['l1_named_ens_only'], medians['l1_named_cat_only']]
l1_pcts   = [v / named_total * 100 if named_total > 0 else 0 for v in l1_vals]
l1_colors = [COLORS['concordant'], COLORS['ensembl_only'], COLORS['cat_only']]
l1_labels = ['Both', 'Ensembl only', 'CAT only']

l2_vals   = [medians['l2_pass'], medians['l2_fail']]
l2_pcts   = [v / medians['l2_total'] * 100 if medians['l2_total'] > 0 else 0 for v in l2_vals]
l2_colors = [COLORS['concordant'], COLORS['discordant']]
l2_labels = ['RBH pass', 'No RBH']

l3_vals  = [medians['l3_full'], medians['l3_partial'], medians['l3_none']]
l3_total = sum(l3_vals)
l3_pcts  = [v / l3_total * 100 if l3_total > 0 else 0 for v in l3_vals]
l3_colors = [COLORS['high_confidence'], COLORS['partial'], COLORS['discordant']]
l3_labels = ['Full match', 'Partial', 'No match']

l3b_vals  = [medians['l3b_protein_coding'], medians['l3b_lncrna'],
             medians['l3b_pseudogene'], medians['l3b_other_ncrna'], medians['l3b_other']]
l3b_total = sum(l3b_vals)
l3b_pcts  = [v / l3b_total * 100 if l3b_total > 0 else 0 for v in l3b_vals]
l3b_colors = ['#1b4f72', '#27ae60', '#8e44ad', '#16a085', '#95a5a6']
l3b_labels = ['Protein-coding', 'lncRNA', 'Pseudogene', 'Other ncRNA', 'Other']

l4_vals  = [medians['l4_intact'], medians['l4_partial'], medians['l4_disrupted']]
l4_total = sum(l4_vals)
l4_pcts  = [v / l4_total * 100 if l4_total > 0 else 0 for v in l4_vals]
l4_colors = [COLORS['high_confidence'], COLORS['partial'], COLORS['discordant']]
l4_labels = ['Intact', 'Partial disruption', 'Disrupted']

# y positions: L1=4, L2=3, L3=2, L3b=1, L4=0
all_levels = [
    (l1_pcts, l1_colors, l1_labels, l1_vals, f'Named genes (n={int(named_total):,})'),
    (l2_pcts, l2_colors, l2_labels, l2_vals, f'All RBH pairs (n={int(medians["l2_total"]):,})'),
    (l3_pcts, l3_colors, l3_labels, l3_vals, f'RBH pass (n={int(l3_total):,})'),
    (l3b_pcts, l3b_colors, l3b_labels, l3b_vals, f'Fully concordant (n={int(l3b_total):,})'),
    (l4_pcts, l4_colors, l4_labels, l4_vals, f'Coding full-concordant (n={int(l4_total):,})'),
]
level_labels = [
    'Named Gene\nPresence',
    'Locus\nOverlap (RBH)',
    'Transcript\nConcordance',
    'Biotype\nBreakdown',
    'CDS\nIntegrity',
]

# ── Draw stacked bars ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
bar_height = 0.6

for i, (pcts, colors, labels, vals, denom_label) in enumerate(all_levels):
    row_y = 4 - i
    left = 0.0
    for pct, color, label, val in zip(pcts, colors, labels, vals):
        ax.barh(row_y, pct, left=left, height=bar_height, color=color,
                edgecolor='white', linewidth=0.5, zorder=1.0)
        if pct > 5:
            ax.text(left + pct / 2, row_y,
                    f'{label}\n{pct:.1f}%\n(n={int(val):,})',
                    ha='center', va='center', fontsize=7, fontweight='bold',
                    color='white', zorder=2.0)
        left += float(pct)
    ax.text(102, row_y, denom_label, va='center', ha='left', fontsize=7.5,
            color='#555555', style='italic', zorder=2.0)

# Divider between L1 (y=4) and L2 (y=3) — L2 has its own denominator
ax.axhline(y=3.5, color='#aaaaaa', linewidth=0.8, linestyle='--', zorder=2.0)
ax.text(50, 3.52, 'rows below are subsets of RBH pairs', ha='center',
        fontsize=7, color='#888888', style='italic', zorder=2.0)

# ── Flow ribbons (each from bottom of upper bar → top of lower bar) ──────────

# L2 → L3: RBH pass distributes into transcript concordance categories
flow_23 = {
    'RBH pass': {'Full match': medians['l3_full'], 'Partial': medians['l3_partial'], 'No match': medians['l3_none']},
    'No RBH':   {'Full match': 0, 'Partial': 0, 'No match': 0},
}
_draw_ribbons_bottom_to_top(
    ax, y_upper_center=3, y_lower_center=2, bar_height=bar_height,
    upper_labels=l2_labels, upper_pcts=l2_pcts,
    lower_labels=l3_labels, lower_pcts=l3_pcts,
    flow_counts=flow_23,
    upper_color_map={'RBH pass': COLORS['concordant'], 'No RBH': COLORS['discordant']},
    alpha=0.22,
)

# L3 → L3b: "Full match" distributes by biotype; Partial/No match do not flow down
flow_33b = {
    'Full match': {lab: v for lab, v in zip(l3b_labels, l3b_vals)},
    'Partial':    {lab: 0 for lab in l3b_labels},
    'No match':   {lab: 0 for lab in l3b_labels},
}
_draw_ribbons_bottom_to_top(
    ax, y_upper_center=2, y_lower_center=1, bar_height=bar_height,
    upper_labels=l3_labels, upper_pcts=l3_pcts,
    lower_labels=l3b_labels, lower_pcts=l3b_pcts,
    flow_counts=flow_33b,
    upper_color_map={l: c for l, c in zip(l3_labels, l3_colors)},
    alpha=0.22,
)

# L3b → L4: only protein-coding flows into CDS integrity
flow_3b4 = {
    'Protein-coding': {'Intact': medians['l4_intact'], 'Partial disruption': medians['l4_partial'], 'Disrupted': medians['l4_disrupted']},
    'lncRNA':         {'Intact': 0, 'Partial disruption': 0, 'Disrupted': 0},
    'Pseudogene':     {'Intact': 0, 'Partial disruption': 0, 'Disrupted': 0},
    'Other ncRNA':    {'Intact': 0, 'Partial disruption': 0, 'Disrupted': 0},
    'Other':          {'Intact': 0, 'Partial disruption': 0, 'Disrupted': 0},
}
_draw_ribbons_bottom_to_top(
    ax, y_upper_center=1, y_lower_center=0, bar_height=bar_height,
    upper_labels=l3b_labels, upper_pcts=l3b_pcts,
    lower_labels=l4_labels, lower_pcts=l4_pcts,
    flow_counts=flow_3b4,
    upper_color_map={l: c for l, c in zip(l3b_labels, l3b_colors)},
    alpha=0.22,
)

# ── Styling ───────────────────────────────────────────────────────────────────
ax.set_yticks(range(5))
ax.set_yticklabels(list(reversed(level_labels)), fontsize=11, fontweight='bold')
ax.set_xlabel('Percentage of row denominator', fontsize=12)
ax.set_xlim(0, 130)
ax.set_title('Ensembl vs CAT Annotation Concordance\n(Median across assemblies)', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main1_sankey.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_main1_sankey.pdf', bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR / 'figure_main1_sankey.png'}")

In [ ]:
# --- Per-assembly distribution strip plot ---
fig, ax = plt.subplots(figsize=(10, 6))

pct_cols = ['l1_named_pct_both', 'l2_pct_pass', 'l3_pct_full', 'l4_pct_intact']
labels = ['Named gene\npresence (both)', 'RBH locus\noverlap', 'Transcript\nconcordance', 'CDS\nintegrity']

for i, (col, label) in enumerate(zip(pct_cols, labels)):
    vals = per_asm[col].dropna()
    jitter = np.random.normal(0, 0.1, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, alpha=0.3, s=8, color=COLORS['concordant'])
    med = vals.median()
    q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
    ax.plot([i-0.3, i+0.3], [med, med], color='black', linewidth=2)
    ax.plot([i, i], [q25, q75], color='black', linewidth=1.5)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Per-assembly concordance at each level', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main1_per_assembly_distribution.png', dpi=300, bbox_inches='tight')
plt.show()